# Successful vs Unsuccessful Grant Comparison

Run the scoring pipeline on all PDFs in `data/successful/` and `data/unsuccessful/`, print results, and export to Excel.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from qwen3_ollama import _Scorer, score_application
from src.all_type_parser.all_type_parser import parse_and_save

CRITERIA_PATH = PROJECT_ROOT / 'criteria_points.json'
SUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'successful'
UNSUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'unsuccessful'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'results' / 'compare'
PARSED_CACHE_DIR = RESULTS_DIR / 'parsed_cache'

EXPERIMENT_OLLAMA_MODEL = 'qwen3.5:27b'

SECTION_KEYS = [
    'general',
    'proposed_research',
    'training_development',
    'sites_support',
    'wpcc',
    'application_form',
]

for path in [RESULTS_DIR, PARSED_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root      :', PROJECT_ROOT)
print('Successful dir    :', SUCCESSFUL_DIR)
print('Unsuccessful dir  :', UNSUCCESSFUL_DIR)
print('Results dir       :', RESULTS_DIR)
print('Model             :', EXPERIMENT_OLLAMA_MODEL)

In [ ]:
def parse_pdf_cached(pdf_path: Path, *, reparse: bool = False) -> tuple[dict, Path]:
    json_path = PARSED_CACHE_DIR / f'{pdf_path.stem}.json'
    if reparse or not json_path.exists():
        parse_and_save(str(pdf_path), str(json_path))
    parsed = json.loads(json_path.read_text(encoding='utf-8'))
    return parsed, json_path


def score_pdf(pdf_path: Path, *, scorer: _Scorer, reparse: bool = False) -> dict:
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    artifact_dir = RESULTS_DIR / 'artifacts' / pdf_path.stem
    artifact_dir.mkdir(parents=True, exist_ok=True)
    result = score_application(
        parsed,
        CRITERIA_PATH,
        doc_id=pdf_path.stem,
        scorer=scorer,
        artifacts_dir=artifact_dir,
    )
    result['source_pdf'] = str(pdf_path)
    result['parsed_json'] = str(parsed_json_path)
    return result


def flatten_row(result: dict, *, pdf_name: str, category: str) -> dict:
    overall = result.get('overall', {})
    row: dict = {
        'pdf_name': pdf_name,
        'category': category,
        'overall_score_100': round(float(overall.get('final_score_0to100', 0)), 2),
    }
    features = result.get('features', {})
    for section_key in SECTION_KEYS:
        sec = features.get(section_key, {}).get('overall', {})
        row[f'{section_key}_score_100'] = round(float(sec.get('final_score_0to100', 0)), 2)
        row[f'{section_key}_evidence_count'] = int(sec.get('evidence_count', 0))
    return row


print('Helper functions defined.')

In [ ]:
scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)

all_pdfs: list[tuple[Path, str]] = [
    *[(p, 'successful') for p in sorted(SUCCESSFUL_DIR.glob('*.pdf'))],
    *[(p, 'unsuccessful') for p in sorted(UNSUCCESSFUL_DIR.glob('*.pdf'))],
]

print(f'PDFs found: {len(all_pdfs)}')
print(f'  successful  : {sum(1 for _, c in all_pdfs if c == "successful")}')
print(f'  unsuccessful: {sum(1 for _, c in all_pdfs if c == "unsuccessful")}')

rows: list[dict] = []
failed: list[tuple[str, str]] = []

for idx, (pdf_path, category) in enumerate(all_pdfs, start=1):
    print(f'\n[{idx}/{len(all_pdfs)}] {pdf_path.name}  [{category}]')
    try:
        result = score_pdf(pdf_path, scorer=scorer)
        row = flatten_row(result, pdf_name=pdf_path.name, category=category)
        rows.append(row)
        print(f'  overall: {row["overall_score_100"]:.1f}/100')
        for section_key in SECTION_KEYS:
            score = row[f'{section_key}_score_100']
            ev = row[f'{section_key}_evidence_count']
            print(f'    {section_key:<25} score={score:5.1f}  evidence={ev}')
    except Exception as exc:
        print(f'  ERROR: {exc}')
        failed.append((pdf_path.name, str(exc)))

print(f'\n\nDone: {len(rows)}/{len(all_pdfs)} succeeded')
if failed:
    print(f'Failed ({len(failed)}):')
    for name, err in failed:
        print(f'  - {name}: {err}')

In [ ]:
df = pd.DataFrame(rows).sort_values(['category', 'pdf_name']).reset_index(drop=True)

# Build column order: pdf_name, category, overall, then for each section: score + evidence
score_cols = [f'{k}_score_100' for k in SECTION_KEYS]
evidence_cols = [f'{k}_evidence_count' for k in SECTION_KEYS]
interleaved = [col for k in SECTION_KEYS for col in (f'{k}_score_100', f'{k}_evidence_count')]
display_cols = ['pdf_name', 'category', 'overall_score_100', *interleaved]

display(df[display_cols])

In [ ]:
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
excel_path = RESULTS_DIR / f'comparison_{timestamp}.xlsx'

df[display_cols].to_excel(excel_path, index=False, sheet_name='Results')

print(f'Exported to: {excel_path}')
print(f'  Rows: {len(df)}  |  Columns: {len(display_cols)}')